In [179]:
import requests
import time
import pandas as pd

In [180]:
ANO_INICIO_BASE = 2024
ANO_ATUAL = 2026

In [181]:
def quali():
    lista_qualis = []
    for i in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
        rodada = 1
        while True:
            acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{i}/{rodada}/qualifying/").json()
            races = acesso["MRData"]["RaceTable"]["Races"]

            if not races:
                break
            else:
                for race in races:
                    qualificatorias = race["QualifyingResults"]
                    circuito = race["Circuit"]["circuitId"]

                    for qualificatoria in qualificatorias:
                        lista_qualis.append({
                            "temporada_atual": acesso["MRData"]["RaceTable"]["season"],
                            "rodada_atual": acesso["MRData"]["RaceTable"]["round"],
                            "id_circuito_atual": circuito,
                            "id_piloto_atual": qualificatoria["Driver"]["driverId"],
                            "id_equipe_atual": qualificatoria["Constructor"]["constructorId"],
                            "posicao_quali_atual": qualificatoria.get("position", None),
                            "q1_atual": qualificatoria.get("Q1", None),
                            "q2_atual": qualificatoria.get("Q2", None),
                            "q3_atual": qualificatoria.get("Q3", None)
                        })
                rodada += 1
            time.sleep(1)
                    
    
    return lista_qualis


qualis = quali()
df = pd.DataFrame(qualis)
df_quali = df.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)


In [182]:
def resultados():
    lista_resultados = []
    for i in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
        rodada = 1
        while True:
            acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{i}/{rodada}/results/").json()
            races = acesso["MRData"]["RaceTable"]["Races"]

            if not races:
                break
            else:
                for race in races:
                    results = race["Results"]
                    circuito = race["Circuit"]["circuitId"]

                    for result in results:
                        lista_resultados.append({
                            "temporada_atual": acesso["MRData"]["RaceTable"]["season"],
                            "rodada_atual": acesso["MRData"]["RaceTable"]["round"],
                            "id_circuito_atual": circuito,
                            "id_piloto_atual": result["Driver"]["driverId"],
                            "id_equipe_atual": result["Constructor"]["constructorId"],
                            "posicao_corrida_anterior": result["positionText"]
                        })
                rodada += 1
            time.sleep(1)
                    
    
    return lista_resultados


results = resultados()
df_results = pd.DataFrame(results)
df_results = df_results.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

In [183]:
df = pd.merge(
    df_quali,
    df_results,
    on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="outer"
)
df["id_circuito_atual"] = df["id_circuito_atual_x"].combine_first(df["id_circuito_atual_y"])
df["id_equipe_atual"] = df["id_equipe_atual_x"].combine_first(df["id_equipe_atual_y"])
df.drop(columns=["id_circuito_atual_x", "id_circuito_atual_y", "id_equipe_atual_x", "id_equipe_atual_y"], inplace=True)
df.reset_index(drop=True,inplace=True)

In [184]:
temporadas = df["temporada_atual"].unique()

In [185]:
ultimo_round = df['rodada_atual']
ultimo_round = ultimo_round.max()


In [186]:
lista_df = []

def converte_ms(ms):
    menor_tempo_rodada = []
    for i in ms:
        if pd.notna(i) and i != '':
            minuto_ms, segundo = i.split(':')
            segundo_ms, milissegundo_ms = segundo.split('.')

            minuto_ms = int(minuto_ms)
            segundo_ms = int(segundo_ms)
            milissegundo_ms = int(milissegundo_ms)

            menor_tempo_rodada.append(((minuto_ms * 60) * 1000) + (segundo_ms * 1000) + milissegundo_ms)
        
    return min(menor_tempo_rodada)


q3_rodada = []
for temporada in temporadas:
    for rodada in range(1, (ultimo_round + 1)):
        df_delta = df[(df['temporada_atual'] == temporada) & (df['rodada_atual'] == rodada)]
        if len(df_delta) > 0:
            menor_tempo = converte_ms(df_delta["q3_atual"])
            for value_q3 in df_delta['q3_atual']:
                if pd.notna(value_q3) and value_q3 != '':
                    dif = converte_ms([value_q3]) - menor_tempo
                    q3_rodada.append(f"{dif:.0f}")
                else:
                    q3_rodada.append(None)

df["dif_para_pole_atual"] = q3_rodada

In [188]:
lista_pontuacao = []

for data in range(ANO_INICIO_BASE, (ANO_ATUAL + 1)):
    rodada = 1
    while True:
        acesso = requests.get(f"https://api.jolpi.ca/ergast/f1/{data}/{rodada}/driverstandings/").json()
        StandingsLists = acesso["MRData"]["StandingsTable"]["StandingsLists"]
        
        if not StandingsLists:
            break
        else:
            for DriverStanding in StandingsLists:
                for pontos in DriverStanding["DriverStandings"]:
                    lista_pontuacao.append({
                        "pontos_anterior": pontos["points"],
                        "rodada_atual": DriverStanding["round"],
                        "temporada_atual": DriverStanding["season"],
                        "id_piloto_atual": pontos["Driver"]["driverId"],
                        "posicao_camp_anterior": pontos["positionText"],
                        "num_vitorias_anterior": pontos["wins"],
                        "media_ultimas_3_anterior": f"{pd.to_numeric(df[(df["temporada_atual"] == data) & (df["rodada_atual"] < rodada) & (df["id_piloto_atual"] == pontos["Driver"]["driverId"])]["posicao_corrida_anterior"], errors="coerce").tail(3).mean():.2f}",
                        "media_ultimas_5_anterior": f"{pd.to_numeric(df[(df["temporada_atual"] == data) & (df["rodada_atual"] < rodada) & (df["id_piloto_atual"] == pontos["Driver"]["driverId"])]["posicao_corrida_anterior"], errors="coerce").tail(5).mean():.2f}",
                    })
            rodada += 1
        time.sleep(1)

df_pontos = pd.DataFrame(lista_pontuacao)
df_pontos = df_pontos.astype({"temporada_atual": int, "rodada_atual": int}).sort_values(by=["temporada_atual", "rodada_atual"], ascending=True)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
total_abandonos = df[(df["temporada_atual"] == 2026) & (df["rodada_atual"] <= 6) & (df["id_piloto_atual"] == "max_verstappen") & (df["posicao_corrida_atual"] == 'R')].count()
print(total_abandonos)

temporada          2
rodada             2
id_piloto          2
posicao_quali      2
q1                 2
q2                 2
q3                 2
posicao_corrida    2
id_circuito        2
id_equipe          2
dif_para_pole      2
dtype: int64


In [ ]:
df["rodada_anterior"] = df["rodada_atual"] - 1

df = pd.merge(
    df,
    df_pontos,
    left_on=["temporada_atual", "rodada_anterior", "id_piloto_atual"],
    right_on=["temporada_atual", "rodada_atual", "id_piloto_atual"],
    how="left"
)

# df.drop(columns=["rodada_anterior", "pontos_anterior_x", "posicao_camp_anterior_x", "num_vitorias_santerior_x", "media_ultimas_3_anterior_x", "media_ultimas_5_anterior_x", "rodada_atual_y"], inplace=True)
# df.rename(columns={'rodada_atual_x': 'rodada_atual', 'pontos_anterior_y': 'pontos_anterior', 'posicao_camp_anterior_y': 'posicao_camp_anterior', 'num_vitorias_anterior_y': 'num_vitorias_anterior', 'media_ultimas_3_anterior_y': 'media_ultimas_3_anterior', 'media_ultimas_5_anterior_y': 'media_ultimas_5_anterior'}, inplace=True)

KeyError: 'rodada_atual'

In [ ]:
# df_ordenado = df.sort_values(by='preco', ascending=False)

df.to_csv("base.csv", index=False)